# Tabela `top3_cliente_trimestre` (gold)

Calcula, por ano e trimestre, os 3 clientes que mais gastaram, com a soma
dos pagamentos de cada um naquele trimestre. Fontes: `fact_pagamento` e
`dim_cliente` (schema `silver`). A escrita é idempotente
(`createOrReplace`), então reexecuções recriam a tabela sem duplicar dados.

In [ ]:
SPARK_APP_NAME = "cria-top3-customer-quarter-gold"

from nucleo import cria_spark_session

spark = cria_spark_session(SPARK_APP_NAME)
spark

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

TABELA_PAGAMENTO = "nessie.silver.fact_pagamento"
TABELA_CLIENTE = "nessie.silver.dim_cliente"
TABELA_DESTINO = "nessie.gold.top3_cliente_trimestre"

janela = Window.partitionBy("ano", "trimestre").orderBy(F.desc("total_gasto"))

df = (
    spark.table(TABELA_PAGAMENTO)
    .withColumn("ano", F.year("data_pagamento"))
    .withColumn("trimestre", F.quarter("data_pagamento"))
    .groupBy("cliente_id", "ano", "trimestre")
    .agg(F.sum("valor").alias("total_gasto"))
    .join(
        spark.table(TABELA_CLIENTE).select(
            "cliente_id",
            F.concat(F.col("nome"), F.lit(" "), F.col("sobrenome")).alias("nome_cliente"),
        ),
        "cliente_id",
    )
    # shortcut: empates no 3o lugar decididos arbitrariamente; dense_rank se o KPI exigir todos
    .withColumn("ranking", F.row_number().over(janela))
    .filter(F.col("ranking") <= 3)
    .select("ano", "trimestre", "ranking", "cliente_id", "nome_cliente", "total_gasto")
)
df.writeTo(TABELA_DESTINO).createOrReplace()

In [ ]:
n = spark.table(TABELA_DESTINO).count()
print(f"{TABELA_DESTINO}: {n} linhas")